<a href="https://colab.research.google.com/github/evandwh/ST554---Spring-2026---NCSU/blob/main/Project_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 2

Author: Evan Whitfield

Course: ST554 - Spring 2026

Purpose: Project 2

In [14]:
from pyspark.sql import DataFrame
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, isnan
from functools import reduce
from pyspark.sql.types import *
import pandas as pd

In [71]:
class SparkDataChecker:
    def __init__(self, dataframe):
        self.df = dataframe

    @classmethod
    def read_csv(cls, spark_session, file_path):
        df = spark.read.load(file_path, format="csv", header=True, inferSchema=True)
        return cls(df)

    @classmethod
    def from_pandas(cls, spark, pandas_df):
        df = spark.createDataFrame(pandas_df)
        return cls(df)

    def column_exists(self, column_name):
        dtype_dict = dict(self.df.dtypes)

        if column_name not in dtype_dict:
            print(f"Column '{column_name}' does not exist.")
            return False

        return True

    def _is_string_column(self, column_name):
        if not self.column_exists(column_name):
            return False

        dtype_dict = dict(self.df.dtypes)

        return dtype_dict[column_name] == 'string'

    def _is_numeric_column(self, column_name):
        numeric_types = ['int', 'bigint', 'double', 'float', 'long', 'decimal', 'longint', 'bigint']

        if not self.column_exists(column_name):
            return False

        dtype_dict = dict(self.df.dtypes)

        return dtype_dict[column_name] in numeric_types

    def check_in_range(self, column_name, lower = None, upper = None):

        if lower is None and upper is None:
            print("Must give at least one of lower or upper.")
            return self

        if not self._is_numeric_column(column_name):
            print(f"Column '{column_name}' is not numeric.")
            return self

        if lower is None:
            lower = float("-inf")

        if upper is None:
            upper = float("inf")

        tf_list = self.df[column_name].between(lower, upper)

        tf_list = when(col(column_name).isNull() | isnan(col(column_name)),
            None).otherwise(tf_list)

        self = self.df.withColumn(f"{column_name}_in_range", tf_list)

        return self

    def str_isin(self, column_name, values):
        if not self._is_string_column(column_name):
            print(f"Column '{column_name}' is not a string.")
            return self

        tf_column = self.df[column_name].isin(values)
        self = self.df.withColumn(f"{column_name}_is_in", tf_column)
        return self

    def NullCheck(self, column_name):
        self = self.df.withColumn(f"{column_name}_is_null", col(column_name).isNull())
        return self

    def summarize_min_max(self, column_name=None, groupedby=None):
        if column_name:
            if not self._is_numeric_column(column_name):
                print(f"'{column_name}’ is not numeric.")
                return None

            if groupedby:
                result = (self.df.groupBy(groupedby).agg(
                    F.min(column_name).alias("min"),
                    F.max(column_name).alias("max")
                    )
                )
            else:
                result = self.df.agg(
                    F.min(column_name).alias("min"),
                    F.max(column_name).alias("max")
                )
            return result.toPandas()

        numeric_cols = [c for c in self.df.columns if self._is_numeric_column(c)]
        if len(numeric_cols) == 0:
            return None

        dfs = []
        for col in numeric_cols:
            if groupedby:
                minmax = (self.df.groupBy(groupedby).agg(
                    F.min(col).alias(f"{col}_min"),
                    F.max(col).alias(f"{col}_max")
                    )
                )
            else:
                minmax = self.df.agg(
                    F.min(col).alias(f"{col}_min"),
                    F.max(col).alias(f"{col}_max")
                )
            dfs.append(minmax)


	      # Merge all results
        if groupedby:
            result = reduce(lambda left, right: left.join(right, on=groupedby), dfs)

        else:
            result = reduce(lambda left, right: left.crossJoin(right), dfs)

        return result.toPandas()



    def summarize_counts(self, col1, col2=None):
        check1 = self._is_string_column(col1)
        check2 = False
        if col2:
            check2 = self._is_string_column(col2)
        # Check first column
        if not check1:
            print(f"'{col1}' is not a string column.")
        if col2 and not check2:
            print(f"'{col2}' is not a string column.")
        if not check1 and not check2:
            return None
        if col2 and not check1:
            result = self.df.groupBy(col2).count()
        elif check1 and check2:
            result = self.df.groupBy(col1, col2).count()
        else:
            result = self.df.groupBy(col1).count()
        return result.toPandas()




In [72]:
data = [
    {"region": "North", "state": "NC", "category": "A", "sub_category": "X", "year": 2023, "sales": 120, "profit": 30},
    {"region": "North", "state": "NC", "category": "A", "sub_category": "Y", "year": 2023, "sales": 200, "profit": 50},
    {"region": "North", "state": "VA", "category": "B", "sub_category": "X", "year": 2023, "sales": 150, "profit": 40},
    {"region": "South", "state": "SC", "category": "A", "sub_category": "X", "year": 2023, "sales": 300, "profit": 70},
    {"region": "South", "state": "GA", "category": "B", "sub_category": "Y", "year": 2023, "sales": 250, "profit": 60},
    {"region": "West", "state": "CA", "category": "A", "sub_category": "X", "year": 2023, "sales": 400, "profit": 90},
    {"region": "West", "state": "CA", "category": "B", "sub_category": "Y", "year": 2023, "sales": 350, "profit": 80},
    {"region": "West", "state": "WA", "category": "A", "sub_category": "X", "year": 2024, "sales": 275, "profit": 65},
    {"region": "North", "state": "NC", "category": "B", "sub_category": "Y", "year": 2024, "sales": 180, "profit": 45},
    {"region": "South", "state": "GA", "category": "A", "sub_category": "X", "year": 2024, "sales": 320, "profit": 75},

    {"region": "North", "state": "VA", "category": "A", "sub_category": "X", "year": 2024, "sales": 210, "profit": 55},
    {"region": "South", "state": "SC", "category": "B", "sub_category": "Y", "year": 2024, "sales": 260, "profit": 60},
    {"region": "West", "state": "WA", "category": "B", "sub_category": "X", "year": 2023, "sales": 310, "profit": 70},
    {"region": "West", "state": "CA", "category": "A", "sub_category": "Y", "year": 2024, "sales": 420, "profit": 95},
    {"region": "North", "state": "NC", "category": "A", "sub_category": "X", "year": 2025, "sales": 190, "profit": 50},
    {"region": "South", "state": "GA", "category": "B", "sub_category": "X", "year": 2025, "sales": 280, "profit": 65},
    {"region": "West", "state": "CA", "category": "B", "sub_category": "Y", "year": 2025, "sales": 390, "profit": 85},
    {"region": "North", "state": "VA", "category": "B", "sub_category": "Y", "year": 2025, "sales": 230, "profit": 60},
    {"region": "South", "state": "SC", "category": "A", "sub_category": "X", "year": 2025, "sales": 340, "profit": 80},
    {"region": "West", "state": "WA", "category": "A", "sub_category": "Y", "year": 2025, "sales": 300, "profit": 70}
]
df = pd.DataFrame(data)

spark = SparkSession.builder.appName("Pandas to Spark").getOrCreate()

data2 = SparkDataChecker.from_pandas(spark, df)

data2.df.show()

data2.summarize_counts("sales", "profit")

+------+-----+--------+------------+----+-----+------+
|region|state|category|sub_category|year|sales|profit|
+------+-----+--------+------------+----+-----+------+
| North|   NC|       A|           X|2023|  120|    30|
| North|   NC|       A|           Y|2023|  200|    50|
| North|   VA|       B|           X|2023|  150|    40|
| South|   SC|       A|           X|2023|  300|    70|
| South|   GA|       B|           Y|2023|  250|    60|
|  West|   CA|       A|           X|2023|  400|    90|
|  West|   CA|       B|           Y|2023|  350|    80|
|  West|   WA|       A|           X|2024|  275|    65|
| North|   NC|       B|           Y|2024|  180|    45|
| South|   GA|       A|           X|2024|  320|    75|
| North|   VA|       A|           X|2024|  210|    55|
| South|   SC|       B|           Y|2024|  260|    60|
|  West|   WA|       B|           X|2023|  310|    70|
|  West|   CA|       A|           Y|2024|  420|    95|
| North|   NC|       A|           X|2025|  190|    50|
| South|  